# Agentic Workflow A

```
MAIN FLOW                           TOOLS

User Prompt
    │
    ▼
┌───────────────┐             ┌───────────────┐
│ News Gathering│────────────►│    SearXNG    │
│     Agent     │◄────────────│               │
└───────────────┘             └───────────────┘
    │
    ▼
┌───────────────┐             ┌───────────────┐
│ News Sentiment│────────────►│    FinBERT    │
│     Agent     │◄────────────│ Sentiment Tool│
└───────────────┘             └───────────────┘
    │
    ▼
┌────────────────┐
│ News Summarizer│
│     Agent      │
└────────────────┘
    │
    ▼
┌────────────────┐
│ News Reporting │
│     Agent      │
└────────────────┘
    │
    ▼
┌─────────────────┐
│   Structured    │
│ Financial Report│
└─────────────────┘
```

## Import Libraries

In [125]:
import torch
import wandb
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from crewai import Agent, Task, Crew, Process
from tqdm import tqdm

## Config

### Project Config

In [126]:
import os, sys

project_path = ".."
sys.path.append(os.path.abspath(project_path))

from utils.tools.search_tool import search_searxng
from utils.tools.sentiment_tool import analyze_sentiment
from utils.tools.markdown_render_tool import markdown_render_tool, render_markdown_report, REPORTS_PATH

reports_path = str(REPORTS_PATH)

## Import Env Variables
from dotenv import load_dotenv
load_dotenv(os.path.join(project_path, ".env"))

## LLM Library
from crewai import LLM

### LLM Config

In [127]:
llm = LLM(
    model="openai/gpt-5-mini"
)

## Tools

In [128]:
from crewai.tools import tool

search_tool = search_searxng

@tool("FinBERT Sentiment Analysis")
def sentiment_tool(text: str, model_type: str = "finbert") -> str:
    """Analyze the sentiment of a news article text using FinBERT or BERT. Returns label and confidence score."""
    return str(analyze_sentiment(text, model_type))

markdown_render_tool = markdown_render_tool

## Create Agents

In [129]:
news_gathering = Agent(
    role="news_gathering",
    goal="Provide news latest articles about stock market and finance prompted by the user. Use the search tool to gather relevant information provide 5 until 10 news articles.",
    backstory="You are a news gathering agent that collects the latest news articles about stock market and finance based on user prompts. You will provide relevant and up-to-date information to assist users in making informed decisions.",
    tools=[search_tool],
    llm=llm,
)

In [130]:
news_sentiment = Agent(
    role="news_sentiment",
    goal="Use the sentiment tool to analyze the sentiment of news articles about stock market and finance prompted by the user. Provide a summary of the sentiment analysis results. the tool would output a sentiment of neutral, positive, or negative.",
    backstory="You are a sentiment analysis agent that analyzes the sentiment of news articles about stock market and finance based on user prompts. You will provide a summary of the sentiment analysis results to assist users in understanding the overall sentiment of the news articles.",
    tools=[sentiment_tool],
    llm=llm,
)

In [131]:
news_summarizing = Agent(
    role="news_summarizing",
    goal="summarize the news articles with the sentiment analysis results and provide a summary of the news articles about stock market and finance prompted by the user.",
    backstory="You are a news gathering agent that collects the latest news articles about stock market and finance based on user prompts. You will provide relevant and up-to-date information to assist users in making informed decisions.",
    llm=llm,
)

In [132]:
news_reporting = Agent(
    role="news_reporting",
    goal="Compile the summarized news articles with sentiment analysis results and provide a final report of the news articles about stock market and finance prompted by the user.",
    backstory="You are a news gathering agent that collects the latest news articles about stock market and finance based on user prompts. You will provide relevant and up-to-date information to assist users in making informed decisions.",
    tools=[markdown_render_tool],
    llm=llm,
)

## Create Tasks

In [133]:
task_gathering = Task(
    description=(
        "Search for the latest news articles about '{topic}' using the search tool. "
        "Collect relevant article titles, URLs, and content snippets worth analyzing."
    ),
    expected_output=(
        "A structured list of the latest relevant news articles about '{topic}', "
        "including title, URL, and content for each article."
    ),
    agent=news_gathering,
)

In [134]:
task_sentiment = Task(
    description=(
        "For each news article gathered about '{topic}', call the analyze_sentiment function "
        "(imported from utils.tools.sentiment_tool) on its text using model_type='finbert'. "
        "Aggregate the results into an overall sentiment summary."
    ),
    expected_output=(
        "A per-article sentiment breakdown (positive/negative/neutral with confidence) "
        "and an overall aggregated sentiment summary for '{topic}'."
    ),
    agent=news_sentiment,
    context=[task_gathering],
)

In [135]:
task_summarizing = Task(
    description=(
        "Summarize the gathered news articles about '{topic}' together with their sentiment analysis results "
        "into a clear, concise summary."
    ),
    expected_output=(
        "A well-structured summary of the latest news about '{topic}' that integrates "
        "the sentiment analysis findings."
    ),
    agent=news_summarizing,
    context=[task_gathering, task_sentiment],
)

In [136]:
task_reporting = Task(
    description=(
        "Compile the summarized news and sentiment results about '{topic}' into a final structured financial report, "
        "then use the markdown_render_tool (topic, summary, sentiment, overview) to save it as a markdown file. "
        "The tool always saves to the project's reports directory — do not pass a path in the filename."
    ),
    expected_output=(
        "A complete structured financial report about '{topic}' in markdown format, saved to disk."
    ),
    agent=news_reporting,
    context=[task_summarizing],
    output_file=os.path.join(reports_path, "financial_report.md"),
)

## Build Crew

In [137]:
crew = Crew(
    agents=[news_gathering, news_sentiment, news_summarizing, news_reporting],
    tasks=[task_gathering, task_sentiment, task_summarizing, task_reporting],
    process=Process.sequential,
    verbose=True,
)

## Run Workflow

In [138]:
topic = "NVIDIA stock"  # change to any stock / finance topic

result = await crew.kickoff_async(inputs={"topic": topic})

print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.22                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b86b188e-ea66-4f71-8269-caf13959d13a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search for the latest news articles about 'NVIDIA stock' using the search tool. Collect relevant         │
│  article titles, URLs, and content snippets worth analyzing.                                                    │
│  ID: 62c9e119-34a5-4d87-a3b5-3a670d4ce452                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: news_gathering                                                                                          │
│                                                                                                                 │
│  Task: Search for the latest news articles about 'NVIDIA stock' using the search tool. Collect relevant         │
│  article titles, URLs, and content snippets worth analyzing.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: sear_xng_search                                                                                          │
│  Args: {'query': 'NVIDIA stock latest news October 2026'}                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#3) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: markdown_report_renderer                                                                                 │
│  Output:                                                                                                        │
│  /run/media/horizontal_roterien_katze/A/Documents/Projects/Python/project_finnessly/reports/nvidia_stock_repor  │
│  t_2026-10-04.md                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: news_reporting                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Below is the complete markdown report saved to disk as reports/nvidia_stock_report_2026-10-04.md               │
│                                                                                                                 │
│  ----                                                                                                           │
│  # NVIDIA stock                                                                                                 │
│                                                                                                                 │
│  ## Summary                                                                                                     │
│  This report compiles eight recent articles about NVIDIA stock (titles, sources, dates, snippets) and FinBERT   │
│  sentiment analysis run on the provided excerpts. Coverage is mixed: 2 positive, 3 neutral, 3 negative. A       │
│  confidence-weighted aggregation produces a slight net-negative signal (weighted average ≈ -0.118 on scale -1   │
│  to +1). Key themes include record-high price/massive market-cap milestones, Q2 FY2027 earnings and corporate   │
│  actions (dividend/buybacks), supply/partner developments (Amazon reportedly offloading $8B of Nvidia chips),   │
│  and analyst commentary on valuation and near-term catalysts.                                                   │
│                                                                                                                 │
│  ## Sentiment (per-article)                                                                                     │
│  Per-article sentiment (label, confidence, raw output):                                                         │
│                                                                                                                 │
│  1) Amazon seeks to offload $8 billion of Nvidia chips to investors, FT reports                                 │
│     - URL:                                                                                                      │
│  https://www.reuters.com/business/retail-consumer/amazon-seeks-offload-8-billion-nvidia-chips-investors-ft-rep  │
│  orts-2026-10-02/                                                                                               │
│     - Date: Oct 1, 2026                                                                                         │
│     - Snippet: Amazon seeks to offload $8B of Nvidia chips to investors, FT reports.                            │
│     - Sentiment: Negative — confidence 99.97%                                                                   │
│     - Raw model output: {'label': 'negative', 'score': 0.9997174143791199}                                      │
│                                                                                                                 │
│  2) Nvidia just hit a record high. Here's why the stock still looks cheap to ...                                │
│     - URL: https://www.cnbc.com/2026/10/02/nvidia-is-a-best-in-breed-stock-on-sale-heres-why-im-buying.html     │
│     - Date: Oct 2, 2026                                                                                         │
│     - Snippet: Nvidia hit a record high; analyst says stock still looks cheap.                                  │
│     - Sentiment: Neutral — confidence 99.14%           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compile the summarized news and sentiment results about 'NVIDIA stock' into a final structured           │
│  financial report, then use the markdown_render_tool (topic, summary, sentiment, overview) to save it as a      │
│  markdown file.                                                                                                 │
│  Agent: news_reporting                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Below is the complete markdown report saved to disk as reports/nvidia_stock_report_2026-10-04.md

----
# NVIDIA stock

## Summary
This report compiles eight recent articles about NVIDIA stock (titles, sources, dates, snippets) and FinBERT sentiment analysis run on the provided excerpts. Coverage is mixed: 2 positive, 3 neutral, 3 negative. A confidence-weighted aggregation produces a slight net-negative signal (weighted average ≈ -0.118 on scale -1 to +1). Key themes include record-high price/massive market-cap milestones, Q2 FY2027 earnings and corporate actions (dividend/buybacks), supply/partner developments (Amazon reportedly offloading $8B of Nvidia chips), and analyst commentary on valuation and near-term catalysts.

## Sentiment (per-article)
Per-article sentiment (label, confidence, raw output):

1) Amazon seeks to offload $8 billion of Nvidia chips to investors, FT reports  
   - URL: https://www.reuters.com/business/retail-consumer/amazon-seeks-offload-8-billion-nvidia-chips-

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b86b188e-ea66-4f71-8269-caf13959d13a                                                                       │
│  Final Output: Below is the complete markdown report saved to disk as                                           │
│  reports/nvidia_stock_report_2026-10-04.md                                                                      │
│                                                                                                                 │
│  ----                                                                                                           │
│  # NVIDIA stock                                                                                                 │
│                                                                                                                 │
│  ## Summary                                                                                                     │
│  This report compiles eight recent articles about NVIDIA stock (titles, sources, dates, snippets) and FinBERT   │
│  sentiment analysis run on the provided excerpts. Coverage is mixed: 2 positive, 3 neutral, 3 negative. A       │
│  confidence-weighted aggregation produces a slight net-negative signal (weighted average ≈ -0.118 on scale -1   │
│  to +1). Key themes include record-high price/massive market-cap milestones, Q2 FY2027 earnings and corporate   │
│  actions (dividend/buybacks), supply/partner developments (Amazon reportedly offloading $8B of Nvidia chips),   │
│  and analyst commentary on valuation and near-term catalysts.                                                   │
│                                                                                                                 │
│  ## Sentiment (per-article)                                                                                     │
│  Per-article sentiment (label, confidence, raw output):                                                         │
│                                                                                                                 │
│  1) Amazon seeks to offload $8 billion of Nvidia chips to investors, FT reports                                 │
│     - URL:                                                                                                      │
│  https://www.reuters.com/business/retail-consumer/amazon-seeks-offload-8-billion-nvidia-chips-investors-ft-rep  │
│  orts-2026-10-02/                                                                                               │
│     - Date: Oct 1, 2026                                                                                         │
│     - Snippet: Amazon seeks to offload $8B of Nvidia chips to investors, FT reports.                            │
│     - Sentiment: Negative — confidence 99.97%                                                                   │
│     - Raw model output: {'label': 'negative', 'score': 0.9997174143791199}                                      │
│                                                                                                                 │
│  2) Nvidia just hit a record high. Here's why the stock still looks cheap to ...                                │
│     - URL: https://www.cnbc.com/2026/10/02/nvidia-is-a-best-in-breed-stock-on-sale-heres-why-im-buying.html     │
│     - Date: Oct 2, 2026                                                                                         │
│     - Snippet: Nvidia hit a record high; analyst says 